Multiple Instance Learning turn-level per classificazione DN4

In [1]:
# 14.1 — Setup
from pathlib import Path

import numpy as np
import pandas as pd

import torch

from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split
)

from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Percorsi
PROJECT_DIR = Path(
    r"C:\Users\acer\Desktop\ProgettoTesi"
)
RESULTS_DIR = (
    PROJECT_DIR
    / "risultati"
)
TURN_PATH = (
    RESULTS_DIR
    / "profilazione_acustica"
    / "turni_con_label_acustica.csv"
)
DN4_PATH = (
    PROJECT_DIR
    / "dati clinici"
    / "patient_id_dn4_score.csv"
)

# Device
DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

# Controllo file
print("\nTURN_PATH esiste:", TURN_PATH.exists())
print("DN4_PATH esiste:", DN4_PATH.exists())

PyTorch: 2.13.0+cpu
Device: cpu

TURN_PATH esiste: True
DN4_PATH esiste: True


In [2]:
# 14.2 — Caricamento dati
turn_df = pd.read_csv(
    TURN_PATH
)
dn4_raw = pd.read_csv(
    DN4_PATH
)
print("TURN DATASET")
print("Shape:", turn_df.shape)
print(
    "Pazienti:",
    turn_df["patient_id"].nunique()
)
print("\nPrime colonne turn:")
print(
    turn_df.columns.tolist()[:30]
)
print("\n" + "=" * 70)
print("DN4 DATASET")
print("Shape:", dn4_raw.shape)
print("\nColonne DN4:")
print(
    dn4_raw.columns.tolist()
)
print("\nPrime righe DN4:")
display(
    dn4_raw.head()
)

TURN DATASET
Shape: (3825, 108)
Pazienti: 90

Prime colonne turn:
['patient_id', 'recording_id', 'nome_file', 'canale', 'turn_index', 'turn_id', 'start_seconds', 'end_seconds', 'turn_duration_seconds', 'patient_speakers', 'n_diar_segments_merged', 'preceding_role', 'preceding_speaker', 'response_latency_seconds', 'audio_path_turn', 'peak_amplitude', 'clipping_ratio', 'f0_p5', 'f0_p25', 'f0_median', 'f0_p75', 'f0_p95', 'f0_mean', 'f0_std', 'f0_iqr', 'f0_range_p95_p5', 'voiced_f0_frames', 'rms_mean', 'rms_std', 'rms_p5']

DN4 DATASET
Shape: (243, 12)

Colonne DN4:
['patient_id', 'dn4_1_1', 'dn4_1_2', 'dn4_1_3', 'dn4_2_4', 'dn4_2_5', 'dn4_2_6', 'dn4_2_7', 'dn4_3_8', 'dn4_3_9', 'dn4_4_10', 'dn4_score']

Prime righe DN4:


,patient_id,dn4_1_1,dn4_1_2,dn4_1_3,dn4_2_4,dn4_2_5,dn4_2_6,dn4_2_7,dn4_3_8,dn4_3_9,dn4_4_10,dn4_score
0,1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,7.0
3,4,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,7.0
4,5,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,5.0


14.3 — Preparazione target DN4

In [ ]:
# Controlli sul file clinico
print("Righe DN4:", len(dn4_raw))
print(
    "Pazienti DN4 unici:",
    dn4_raw["patient_id"].nunique()
)

print(
    "Duplicati patient_id:",
    dn4_raw["patient_id"].duplicated().sum()
)

print(
    "DN4 score mancanti:",
    dn4_raw["dn4_score"].isna().sum()
)

# Manteniamo solo patient_id e score totale
dn4_patient = (
    dn4_raw[
        [
            "patient_id",
            "dn4_score"
        ]
    ]
    .drop_duplicates("patient_id")
    .copy()
)

# Classificazione binaria
#
# DN4 < 4  -> negativo
# DN4 >= 4 -> positivo
dn4_patient["dn4_class"] = np.where(
    dn4_patient["dn4_score"] >= 4,
    "positivo",
    "negativo"
)
dn4_patient["dn4_label"] = (
    dn4_patient["dn4_class"]
    .map({
        "negativo": 0,
        "positivo": 1
    })
    .astype(int)
)

# Merge con i 3825 turni
mil_df = turn_df.merge(
    dn4_patient,
    on="patient_id",
    how="inner",
    validate="many_to_one"
)
print("\n" + "=" * 70)
print("DATASET MIL DN4")
print("=" * 70)
print("Turni:", len(mil_df))
print(
    "Pazienti:",
    mil_df["patient_id"].nunique()
)
print(
    "DN4 score mancanti dopo merge:",
    mil_df["dn4_score"].isna().sum()
)

# Distribuzione PATIENT-LEVEL
patient_mil = (
    mil_df[
        [
            "patient_id",
            "dn4_score",
            "dn4_class",
            "dn4_label"
        ]
    ]
    .drop_duplicates("patient_id")
    .sort_values("patient_id")
    .reset_index(drop=True)
)
print("\nDistribuzione DN4 patient-level:")
print(
    patient_mil["dn4_class"]
    .value_counts()
)
print("\nStatistiche DN4:")
display(
    patient_mil["dn4_score"]
    .describe()
)
# Controlli attesi
assert mil_df["patient_id"].nunique() == 90
assert patient_mil["patient_id"].nunique() == 90
assert mil_df["dn4_score"].isna().sum() == 0

print("mil_df esiste:", "mil_df" in globals())
print("Shape:", mil_df.shape)

Righe DN4: 243
Pazienti DN4 unici: 243
Duplicati patient_id: 0
DN4 score mancanti: 4

DATASET MIL DN4
Turni: 3825
Pazienti: 90
DN4 score mancanti dopo merge: 0

Distribuzione DN4 patient-level:
dn4_class
positivo    46
negativo    44
Name: count, dtype: int64

Statistiche DN4:


count    90.000000
mean      3.500000
std       2.757014
min       0.000000
25%       1.000000
50%       4.000000
75%       5.750000
max       9.000000
Name: dn4_score, dtype: float64

mil_df esiste: True
Shape: (3825, 111)


14.3b — Controllo finale merge DN4

In [5]:
print("Turni:", len(mil_df))
print("Pazienti:", mil_df["patient_id"].nunique())
print("\nDistribuzione DN4 patient-level:")
display(
    mil_df[
        ["patient_id", "dn4_class"]
    ]
    .drop_duplicates("patient_id")
    ["dn4_class"]
    .value_counts()
)

print(
    "\nDN4 mancanti:",
    mil_df["dn4_score"].isna().sum()
)

Turni: 3825
Pazienti: 90

Distribuzione DN4 patient-level:


dn4_class
positivo    46
negativo    44
Name: count, dtype: int64


DN4 mancanti: 0


14.4 — Feature candidate per MIL Stesse feature del Notebook 13

In [6]:
FEATURE_COLS_MIL = [

    # Ampiezza
    "peak_amplitude",
    "clipping_ratio",

    # F0
    "f0_median",
    "f0_std",
    "f0_iqr",
    "f0_range_p95_p5",

    # Energia
    "rms_mean",
    "rms_std",
    "rms_p5",
    "rms_p95",

    # Speech activity / pause
    "speech_activity_ratio",
    "internal_pause_mean_seconds",
    "internal_pause_median_seconds",
    "internal_pause_max_seconds",
    "internal_pause_rate_per_min",

    # MFCC — mean e std
    "mfcc_1_mean", "mfcc_1_std",
    "mfcc_2_mean", "mfcc_2_std",
    "mfcc_3_mean", "mfcc_3_std",
    "mfcc_4_mean", "mfcc_4_std",
    "mfcc_5_mean", "mfcc_5_std",
    "mfcc_6_mean", "mfcc_6_std",
    "mfcc_7_mean", "mfcc_7_std",
    "mfcc_8_mean", "mfcc_8_std",
    "mfcc_9_mean", "mfcc_9_std",
    "mfcc_10_mean", "mfcc_10_std",
    "mfcc_11_mean", "mfcc_11_std",
    "mfcc_12_mean", "mfcc_12_std",
    "mfcc_13_mean", "mfcc_13_std",

    # Spectral descriptors
    "centroid_mean",
    "centroid_std",
    "rolloff_mean",
    "rolloff_std",
    "bandwidth_mean",
    "bandwidth_std",
    "flatness_mean",
    "flatness_std",
    "zcr_mean",
    "zcr_std",

    # Spectral contrast
    "contrast_1_mean", "contrast_1_std",
    "contrast_2_mean", "contrast_2_std",
    "contrast_3_mean", "contrast_3_std",
    "contrast_4_mean", "contrast_4_std",
    "contrast_5_mean", "contrast_5_std",
    "contrast_6_mean", "contrast_6_std",
    "contrast_7_mean", "contrast_7_std",

    # Voice quality
    "hnr",
    "jitter",
    "shimmer",

    # Trascrizione / comportamento
    "words_per_second",
    "articulation_words_per_second",
    "filler_rate_per_100_words",
    "repetition_rate_per_100_words"
]

print(
    "Numero feature MIL:",
    len(FEATURE_COLS_MIL)
)

# Controllo che esistano tutte
missing_features = [
    col
    for col in FEATURE_COLS_MIL
    if col not in mil_df.columns
]
print(
    "Feature mancanti:",
    missing_features
)

# Missingness
feature_missingness = (
    mil_df[FEATURE_COLS_MIL]
    .isna()
    .mean()
    .mul(100)
    .sort_values(
        ascending=False
    )
)
print("\nFeature con maggiore missingness:")

display(
    feature_missingness.head(15)
    .to_frame("missing_percent")
    .round(2)
)

assert len(FEATURE_COLS_MIL) == 72
assert len(missing_features) == 0

Numero feature MIL: 72
Feature mancanti: []

Feature con maggiore missingness:


,missing_percent
f0_std,6.82
f0_median,6.82
f0_range_p95_p5,6.82
f0_iqr,6.82
filler_rate_per_100_words,5.46
repetition_rate_per_100_words,5.46
shimmer,0.05
peak_amplitude,0.00
rms_std,0.00
rms_mean,0.00


14.5 — Dimensione delle bag

In [7]:
bag_sizes = (
    mil_df
    .groupby("patient_id")
    .size()
    .rename("n_turns")
)

print("Numero bag:", len(bag_sizes))

print("\nStatistiche turni per paziente:")
display(
    bag_sizes.describe()
)

bag_summary = (
    patient_mil
    .merge(
        bag_sizes.reset_index(),
        on="patient_id",
        how="left",
        validate="one_to_one"
    )
)

print("\nTurni per classe DN4:")
display(
    bag_summary
    .groupby("dn4_class")["n_turns"]
    .describe()
    .round(2)
)

Numero bag: 90

Statistiche turni per paziente:


count    90.000000
mean     42.500000
std      17.690377
min       8.000000
25%      31.250000
50%      39.000000
75%      51.750000
max      98.000000
Name: n_turns, dtype: float64


Turni per classe DN4:


,count,mean,std,min,25%,50%,75%,max
dn4_class,,,,,,,,
negativo,44.0,39.98,17.54,8.0,30.5,39.0,48.5,98.0
positivo,46.0,44.91,17.68,17.0,32.0,41.0,53.0,93.0


14.6 — 5-fold stratificati a livello PAZIENTE

In [8]:
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 5
RANDOM_STATE = 42

# Una sola riga per paziente
mil_patients = (
    patient_mil[
        [
            "patient_id",
            "dn4_score",
            "dn4_class",
            "dn4_label"
        ]
    ]
    .sort_values("patient_id")
    .reset_index(drop=True)
    .copy()
)

# StratifiedKFold
skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)
mil_patients["fold"] = -1

for fold, (_, test_idx) in enumerate(
    skf.split(
        mil_patients["patient_id"],
        mil_patients["dn4_label"]
    )
):
    mil_patients.loc[
        test_idx,
        "fold"
    ] = fold
mil_patients["fold"] = (
    mil_patients["fold"]
    .astype(int)
)

# Controlli
print("Pazienti totali:", len(mil_patients))
print(
    "Pazienti unici:",
    mil_patients["patient_id"].nunique()
)
print(
    "Fold presenti:",
    sorted(
        mil_patients["fold"].unique()
    )
)
print("\nDistribuzione pazienti per fold:")

fold_class_table = pd.crosstab(
    mil_patients["fold"],
    mil_patients["dn4_class"]
)

display(
    fold_class_table
)

print("\nNumero pazienti per fold:")

display(
    mil_patients
    .groupby("fold")
    .size()
    .rename("n_patients")
    .to_frame()
)

# Verifica train/test per ciascun fold
for fold in range(N_FOLDS):
    train_ids = set(
        mil_patients.loc[
            mil_patients["fold"] != fold,
            "patient_id"
        ]
    )
    test_ids = set(
        mil_patients.loc[
            mil_patients["fold"] == fold,
            "patient_id"
        ]
    )
    overlap = (
        train_ids
        & test_ids
    )
    print(
        f"Fold {fold}: "
        f"train={len(train_ids)} | "
        f"test={len(test_ids)} | "
        f"overlap={len(overlap)}"
    )
    assert len(overlap) == 0

# Controlli finali
assert len(mil_patients) == 90
assert mil_patients["patient_id"].nunique() == 90
assert set(mil_patients["fold"]) == {0, 1, 2, 3, 4}

Pazienti totali: 90
Pazienti unici: 90
Fold presenti: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Distribuzione pazienti per fold:


dn4_class,negativo,positivo
fold,,
0,9,9
1,9,9
2,9,9
3,9,9
4,8,10



Numero pazienti per fold:


,n_patients
fold,
0,18
1,18
2,18
3,18
4,18


Fold 0: train=72 | test=18 | overlap=0
Fold 1: train=72 | test=18 | overlap=0
Fold 2: train=72 | test=18 | overlap=0
Fold 3: train=72 | test=18 | overlap=0
Fold 4: train=72 | test=18 | overlap=0


14.6b — Fold associato a ciascun turno

In [9]:
mil_df = (
    mil_df
    .drop(
        columns=["fold"],
        errors="ignore"
    )
    .merge(
        mil_patients[
            [
                "patient_id",
                "fold"
            ]
        ],
        on="patient_id",
        how="left",
        validate="many_to_one"
    )
)
print(
    "Turni totali:",
    len(mil_df)
)
print(
    "Pazienti:",
    mil_df["patient_id"].nunique()
)
print("\nTurni presenti in ciascun test fold:")

display(
    mil_df
    .groupby("fold")
    .agg(
        n_turns=("patient_id", "size"),
        n_patients=("patient_id", "nunique")
    )
)

assert mil_df["fold"].isna().sum() == 0
assert len(mil_df) == 3825
assert mil_df["patient_id"].nunique() == 90

Turni totali: 3825
Pazienti: 90

Turni presenti in ciascun test fold:


,n_turns,n_patients
fold,,
0,755,18
1,734,18
2,814,18
3,789,18
4,733,18


14.7 — Preprocessing + costruzione bag MIL

In [10]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

import torch
import numpy as np
import pandas as pd

# Fit del preprocessing
def fit_mil_preprocessor(
    train_df,
    feature_cols
):
    """
    Fit di imputazione e scaling SOLO sul training.
    """

    imputer = SimpleImputer(
        strategy="median"
    )
    X_train_imp = imputer.fit_transform(
        train_df[feature_cols]
    )
    scaler = RobustScaler()
    scaler.fit(
        X_train_imp
    )
    return imputer, scaler

# Applicazione preprocessing
def transform_mil_dataframe(
    df,
    feature_cols,
    imputer,
    scaler
):
    """
    Trasforma un dataframe usando imputer e scaler
    già fittati sul training.
    """

    transformed = df.copy()
    X_imp = imputer.transform(
        transformed[feature_cols]
    )
    X_scaled = scaler.transform(
        X_imp
    )
    transformed.loc[
        :,
        feature_cols
    ] = X_scaled
    return transformed

# Costruzione delle bag
def build_patient_bags(
    df,
    feature_cols
):
    """
    Ogni paziente = una bag.
    Ogni riga/turno = un'istanza della bag.
    """

    bags = []
    for patient_id, group in (
        df
        .groupby("patient_id", sort=True)
    ):
        # Una sola label per paziente
        labels = (
            group["dn4_label"]
            .unique()
        )
        assert len(labels) == 1
        label = int(
            labels[0]
        )
        # Matrice: n_turni x n_feature
        X = (
            group[feature_cols]
            .to_numpy(
                dtype=np.float32
            )
        )
        X_tensor = torch.tensor(
            X,
            dtype=torch.float32
        )
        y_tensor = torch.tensor(
            label,
            dtype=torch.float32
        )
        bags.append({
            "patient_id": patient_id,
            "X": X_tensor,
            "y": y_tensor,
            "n_turns": len(group)
        })
    return bags

print(
    "Funzioni definite correttamente."
)

Funzioni definite correttamente.


14.8 — Smoke test fold 0

In [11]:
TEST_FOLD = 0

train_fold0 = (
    mil_df[
        mil_df["fold"] != TEST_FOLD
    ]
    .copy()
)
test_fold0 = (
    mil_df[
        mil_df["fold"] == TEST_FOLD
    ]
    .copy()
)
print(
    "Train pazienti:",
    train_fold0["patient_id"].nunique()
)
print(
    "Test pazienti:",
    test_fold0["patient_id"].nunique()
)

# Fit SOLO training
imputer_test, scaler_test = (
    fit_mil_preprocessor(
        train_fold0,
        FEATURE_COLS_MIL
    )
)

# Trasformazione
train_fold0_scaled = (
    transform_mil_dataframe(
        train_fold0,
        FEATURE_COLS_MIL,
        imputer_test,
        scaler_test
    )
)
test_fold0_scaled = (
    transform_mil_dataframe(
        test_fold0,
        FEATURE_COLS_MIL,
        imputer_test,
        scaler_test
    )
)

# Costruzione bag
train_bags_test = (
    build_patient_bags(
        train_fold0_scaled,
        FEATURE_COLS_MIL
    )
)
test_bags_test = (
    build_patient_bags(
        test_fold0_scaled,
        FEATURE_COLS_MIL
    )
)
print("\nBag training:", len(train_bags_test))
print("Bag test:", len(test_bags_test))

# Controlliamo la prima bag
example_bag = train_bags_test[0]
print("\nPrima bag:")
print(
    "patient_id:",
    example_bag["patient_id"]
)
print(
    "Shape X:",
    example_bag["X"].shape
)
print(
    "Label:",
    example_bag["y"].item()
)
print(
    "Numero turni:",
    example_bag["n_turns"]
)

# Controlli
assert len(train_bags_test) == 72
assert len(test_bags_test) == 18
assert example_bag["X"].shape[1] == 72
assert not torch.isnan(
    example_bag["X"]
).any()
assert not torch.isinf(
    example_bag["X"]
).any()

Train pazienti: 72
Test pazienti: 18

Bag training: 72
Bag test: 18

Prima bag:
patient_id: 1
Shape X: torch.Size([34, 72])
Label: 0.0
Numero turni: 34


14.9 — Dataset MIL + padding delle bag

In [12]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import torch

class PatientBagDataset(Dataset):
    def __init__(self, bags):
        self.bags = bags
    def __len__(self):
        return len(self.bags)
    def __getitem__(self, idx):

        bag = self.bags[idx]
        return {
            "patient_id": bag["patient_id"],
            "X": bag["X"],
            "y": bag["y"],
            "n_turns": bag["n_turns"]
        }

def mil_collate_fn(batch):
    """
    Crea un batch di pazienti con numero variabile di turni.

    X padded:
        [batch_size, max_n_turns, n_features]

    mask:
        True  = turno reale
        False = padding
    """

    X_list = [
        item["X"]
        for item in batch
    ]

    lengths = torch.tensor(
        [
            item["X"].shape[0]
            for item in batch
        ],
        dtype=torch.long
    )

    # Padding a zero
    X_padded = pad_sequence(
        X_list,
        batch_first=True,
        padding_value=0.0
    )
    batch_size = len(batch)
    max_length = X_padded.shape[1]

    # Mask delle istanze REALI
    positions = (
        torch.arange(max_length)
        .unsqueeze(0)
        .expand(batch_size, max_length)
    )
    mask = (
        positions
        < lengths.unsqueeze(1)
    )

    # Label patient-level
    y = torch.tensor(
        [
            int(item["y"].item())
            for item in batch
        ],
        dtype=torch.long
    )
    patient_ids = [
        item["patient_id"]
        for item in batch
    ]
    return {
        "patient_id": patient_ids,
        "X": X_padded,
        "mask": mask,
        "y": y,
        "lengths": lengths
    }
print("PatientBagDataset definito.")
print("mil_collate_fn definita.")

PatientBagDataset definito.
mil_collate_fn definita.


14.10 — Gated Attention MIL

In [13]:
import torch
import torch.nn as nn


class GatedAttentionMIL(nn.Module):
    def __init__(
        self,
        input_dim=72,
        instance_dim=64,
        attention_dim=32,
        classifier_dim=32,
        dropout=0.2
    ):

        super().__init__()

        # 1. INSTANCE ENCODER
        # Ogni turno:
        # 72 feature -> rappresentazione 64D
        self.instance_encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                instance_dim
            ),
            nn.LayerNorm(
                instance_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            )
        )

        # 2. GATED ATTENTION
        # ramo V: tanh
        # ramo U: sigmoid
        self.attention_V = nn.Linear(
            instance_dim,
            attention_dim
        )
        self.attention_U = nn.Linear(
            instance_dim,
            attention_dim
        )
        self.attention_w = nn.Linear(
            attention_dim,
            1,
            bias=False
        )

        # 3. CLASSIFICATORE PATIENT-LEVEL
        self.classifier = nn.Sequential(
            nn.Linear(
                instance_dim,
                classifier_dim
            ),
            nn.LayerNorm(
                classifier_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                classifier_dim,
                2
            )
        )
    def forward(
        self,
        X,
        mask
    ):

        """
        X:
            [B, N, 72]

        mask:
            [B, N]

        output:
            logits      [B, 2]
            attention   [B, N]
        """

        # Instance encoding
        H = self.instance_encoder(
            X
        )
        # H:
        # [B, N, 64]

        # Gated attention
        V = torch.tanh(
            self.attention_V(H)
        )
        U = torch.sigmoid(
            self.attention_U(H)
        )
        gated = V * U
        attention_scores = (
            self.attention_w(
                gated
            )
            .squeeze(-1)
        )
        # [B, N]

        # Il padding NON deve ricevere attention
        attention_scores = (
            attention_scores
            .masked_fill(
                ~mask,
                -1e9
            )
        )

        # Softmax tra i turni del paziente
        attention_weights = (
            torch.softmax(
                attention_scores,
                dim=1
            )
        )

        # ulteriore sicurezza
        attention_weights = (
            attention_weights
            * mask.float()
        )
        attention_weights = (
            attention_weights
            /
            attention_weights
            .sum(
                dim=1,
                keepdim=True
            )
            .clamp_min(1e-8)
        )

        # Weighted pooling
        # da N turni -> UN embedding paziente
        patient_embedding = (
            torch.bmm(
                attention_weights.unsqueeze(1),
                H
            )
            .squeeze(1)
        )
        # [B, 64]

        # Classificazione DN4
        logits = self.classifier(
            patient_embedding
        )
        # [B, 2]

        return (
            logits,
            attention_weights
        )
print("GatedAttentionMIL definito.")

GatedAttentionMIL definito.


14.11 — Smoke test Gated Attention MIL

In [ ]:
test_dataset = PatientBagDataset(
    train_bags_test
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=mil_collate_fn
)

# Prendiamo un solo batch
batch = next(
    iter(test_loader)
)
print("Patient IDs:")
print(
    batch["patient_id"]
)
print(
    "\nShape X padded:",
    batch["X"].shape
)
print(
    "Shape mask:",
    batch["mask"].shape
)
print(
    "Label:",
    batch["y"]
)
print(
    "Lunghezze originali:",
    batch["lengths"]
)

# Modello
model_test = GatedAttentionMIL(
    input_dim=len(FEATURE_COLS_MIL),
    instance_dim=64,
    attention_dim=32,
    classifier_dim=32,
    dropout=0.2
).to(DEVICE)

X_batch = (
    batch["X"]
    .to(DEVICE)
)
mask_batch = (
    batch["mask"]
    .to(DEVICE)
)

model_test.eval()

with torch.no_grad():
    logits, attention = (
        model_test(
            X_batch,
            mask_batch
        )
    )
print(
    "\nShape logits:",
    logits.shape
)
print(
    "Shape attention:",
    attention.shape
)

# Probabilità DN4 positivo
probabilities = torch.softmax(
    logits,
    dim=1
)
print(
    "\nProbabilità:"
)
print(
    probabilities.cpu()
)

# Controllo fondamentale: i pesi attention dei turni reali devono sommare a 1
attention_sums = (
    attention
    .sum(dim=1)
    .cpu()
)
print(
    "\nSomma attention per paziente:"
)
print(
    attention_sums
)

# Padding deve avere peso zero
padding_attention = (
    attention[
        ~mask_batch
    ]
)

if padding_attention.numel() > 0:
    print(
        "\nMassima attention sul padding:",
        padding_attention
        .abs()
        .max()
        .item()
    )

# Assertions
assert logits.shape == (4, 2)
assert attention.shape[0] == 4
assert torch.allclose(
    attention.sum(dim=1),
    torch.ones(
        4,
        device=DEVICE
    ),
    atol=1e-5
)
if padding_attention.numel() > 0:
    assert torch.all(
        padding_attention == 0
    )
print(
    "\nSmoke test modello MIL completato correttamente."
)

Patient IDs:
[1, 2, 3, 10]

Shape X padded: torch.Size([4, 34, 72])
Shape mask: torch.Size([4, 34])
Label: tensor([0, 0, 1, 1])
Lunghezze originali: tensor([34, 33, 28, 22])

Shape logits: torch.Size([4, 2])
Shape attention: torch.Size([4, 34])

Probabilità:
tensor([[0.4653, 0.5347],
        [0.5184, 0.4816],
        [0.5164, 0.4836],
        [0.3851, 0.6149]])

Somma attention per paziente:
tensor([1.0000, 1.0000, 1.0000, 1.0000])

Massima attention sul padding: 0.0

Smoke test modello MIL completato correttamente.


14.12 — Inner split patient-level sul fold 0

In [15]:
from sklearn.model_selection import train_test_split
import numpy as np
import random
import torch

SEED = 42
VAL_SIZE = 0.20


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(SEED)

OUTER_FOLD = 0

# Pazienti disponibili per il training dell'outer fold 0
outer_train_patients = (
    mil_patients[
        mil_patients["fold"] != OUTER_FOLD
    ]
    .copy()
    .reset_index(drop=True)
)
outer_test_patients = (
    mil_patients[
        mil_patients["fold"] == OUTER_FOLD
    ]
    .copy()
    .reset_index(drop=True)
)

# Inner train / validation
# Stratificazione DN4
inner_train_patients, val_patients = train_test_split(
    outer_train_patients,
    test_size=VAL_SIZE,
    stratify=outer_train_patients["dn4_label"],
    random_state=SEED
)
print("Outer train:", len(outer_train_patients))
print("Outer test:", len(outer_test_patients))
print("\nInner train:", len(inner_train_patients))
print("Validation:", len(val_patients))
print("\nDistribuzione INNER TRAIN:")
print(
    inner_train_patients["dn4_class"]
    .value_counts()
)
print("\nDistribuzione VALIDATION:")
print(
    val_patients["dn4_class"]
    .value_counts()
)
print("\nDistribuzione OUTER TEST:")
print(
    outer_test_patients["dn4_class"]
    .value_counts()
)

# Nessun paziente deve comparire in più gruppi
train_ids = set(
    inner_train_patients["patient_id"]
)
val_ids = set(
    val_patients["patient_id"]
)
test_ids = set(
    outer_test_patients["patient_id"]
)
print(
    "\nOverlap train-val:",
    len(train_ids & val_ids)
)
print(
    "Overlap train-test:",
    len(train_ids & test_ids)
)
print(
    "Overlap val-test:",
    len(val_ids & test_ids)
)
assert len(train_ids & val_ids) == 0
assert len(train_ids & test_ids) == 0
assert len(val_ids & test_ids) == 0

Outer train: 72
Outer test: 18

Inner train: 57
Validation: 15

Distribuzione INNER TRAIN:
dn4_class
positivo    29
negativo    28
Name: count, dtype: int64

Distribuzione VALIDATION:
dn4_class
positivo    8
negativo    7
Name: count, dtype: int64

Distribuzione OUTER TEST:
dn4_class
positivo    9
negativo    9
Name: count, dtype: int64

Overlap train-val: 0
Overlap train-test: 0
Overlap val-test: 0


14.13 — Preparazione bag train / validation / test

In [16]:
inner_train_df = mil_df[
    mil_df["patient_id"].isin(
        inner_train_patients["patient_id"]
    )
].copy()
val_df = mil_df[
    mil_df["patient_id"].isin(
        val_patients["patient_id"]
    )
].copy()
outer_test_df = mil_df[
    mil_df["patient_id"].isin(
        outer_test_patients["patient_id"]
    )
].copy()

# Fit preprocessing SOLO inner training
imputer_fold0, scaler_fold0 = (
    fit_mil_preprocessor(
        inner_train_df,
        FEATURE_COLS_MIL
    )
)

# Transform
inner_train_scaled = transform_mil_dataframe(
    inner_train_df,
    FEATURE_COLS_MIL,
    imputer_fold0,
    scaler_fold0
)
val_scaled = transform_mil_dataframe(
    val_df,
    FEATURE_COLS_MIL,
    imputer_fold0,
    scaler_fold0
)
outer_test_scaled = transform_mil_dataframe(
    outer_test_df,
    FEATURE_COLS_MIL,
    imputer_fold0,
    scaler_fold0
)

# Bag
inner_train_bags = build_patient_bags(
    inner_train_scaled,
    FEATURE_COLS_MIL
)
val_bags = build_patient_bags(
    val_scaled,
    FEATURE_COLS_MIL
)
outer_test_bags = build_patient_bags(
    outer_test_scaled,
    FEATURE_COLS_MIL
)
print("Bag inner train:", len(inner_train_bags))
print("Bag validation:", len(val_bags))
print("Bag outer test:", len(outer_test_bags))

assert len(inner_train_bags) == len(inner_train_patients)
assert len(val_bags) == len(val_patients)
assert len(outer_test_bags) == 18

Bag inner train: 57
Bag validation: 15
Bag outer test: 18


14.14 — DataLoader pilot fold 0

In [17]:
BATCH_SIZE = 8

train_loader = DataLoader(
    PatientBagDataset(inner_train_bags),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=mil_collate_fn,
    num_workers=0
)

val_loader = DataLoader(
    PatientBagDataset(val_bags),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=mil_collate_fn,
    num_workers=0
)

test_loader = DataLoader(
    PatientBagDataset(outer_test_bags),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=mil_collate_fn,
    num_workers=0
)


print("Batch training:", len(train_loader))
print("Batch validation:", len(val_loader))
print("Batch test:", len(test_loader))

batch = next(iter(train_loader))

print("\nShape primo batch:")
print("X:", batch["X"].shape)
print("mask:", batch["mask"].shape)
print("y:", batch["y"].shape)

Batch training: 8
Batch validation: 2
Batch test: 3

Shape primo batch:
X: torch.Size([8, 64, 72])
mask: torch.Size([8, 64])
y: torch.Size([8])


14.15 — Pesi di classe patient-level

In [18]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch


inner_y = (
    inner_train_patients["dn4_label"]
    .to_numpy()
)

class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=inner_y
)

class_weights = torch.tensor(
    class_weights_np,
    dtype=torch.float32,
    device=DEVICE
)

print(
    "Pesi classe [negativo, positivo]:",
    class_weights.cpu().numpy()
)

Pesi classe [negativo, positivo]: [1.0178572  0.98275864]


14.16 — Funzioni training / evaluation MIL

In [19]:
import copy
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):
    model.train()

    total_loss = 0.0
    total_patients = 0

    for batch in loader:

        X = batch["X"].to(device)
        mask = batch["mask"].to(device)
        y = batch["y"].to(device)

        optimizer.zero_grad()

        logits, _ = model(
            X,
            mask
        )

        loss = criterion(
            logits,
            y
        )

        loss.backward()

        # protezione da gradienti troppo grandi
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        batch_size = y.shape[0]

        total_loss += (
            loss.item()
            * batch_size
        )

        total_patients += batch_size

    return (
        total_loss
        / total_patients
    )

@torch.no_grad()
def evaluate_mil(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    total_patients = 0
    patient_ids = []
    y_true = []
    prob_positive = []

    for batch in loader:
        X = batch["X"].to(device)
        mask = batch["mask"].to(device)
        y = batch["y"].to(device)
        logits, attention = model(
            X,
            mask
        )

        loss = criterion(
            logits,
            y
        )

        probs = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        batch_size = y.shape[0]

        total_loss += (
            loss.item()
            * batch_size
        )

        total_patients += batch_size

        patient_ids.extend(
            batch["patient_id"]
        )

        y_true.extend(
            y.cpu()
            .numpy()
            .tolist()
        )

        prob_positive.extend(
            probs.cpu()
            .numpy()
            .tolist()
        )

    y_true = np.asarray(
        y_true,
        dtype=int
    )

    prob_positive = np.asarray(
        prob_positive,
        dtype=float
    )

    y_pred = (
        prob_positive >= 0.5
    ).astype(int)

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred
        )
    )

    f1 = f1_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    # AUC solo se entrambe le classi
    # sono presenti
    if len(np.unique(y_true)) == 2:

        auc = roc_auc_score(
            y_true,
            prob_positive
        )

    else:
        auc = np.nan

    return {
        "loss":
            total_loss / total_patients,

        "accuracy":
            accuracy,

        "balanced_accuracy":
            balanced_accuracy,

        "f1":
            f1,

        "roc_auc":
            auc,

        "patient_id":
            patient_ids,

        "y_true":
            y_true,

        "y_pred":
            y_pred,

        "prob_positive":
            prob_positive
    }

print(
    "Funzioni training/evaluation definite."
)

Funzioni training/evaluation definite.


14.17 — Pilot MIL fold 0 Early stopping su validation

In [20]:
set_seed(42)

# Modello
pilot_model = GatedAttentionMIL(
    input_dim=len(FEATURE_COLS_MIL),
    instance_dim=64,
    attention_dim=32,
    classifier_dim=32,
    dropout=0.2
).to(DEVICE)

# Loss
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# Optimizer
optimizer = torch.optim.AdamW(
    pilot_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# Early stopping
MAX_EPOCHS = 200
PATIENCE = 25
MIN_DELTA = 1e-4

best_val_loss = np.inf
best_epoch = 0
best_state = None

epochs_without_improvement = 0

history = []

# Training
for epoch in range(
    1,
    MAX_EPOCHS + 1
):

    train_loss = train_one_epoch(
        pilot_model,
        train_loader,
        optimizer,
        criterion,
        DEVICE
    )
    val_metrics = evaluate_mil(
        pilot_model,
        val_loader,
        criterion,
        DEVICE
    )
    history.append({
        "epoch": epoch,

        "train_loss":
            train_loss,

        "val_loss":
            val_metrics["loss"],

        "val_accuracy":
            val_metrics["accuracy"],

        "val_balanced_accuracy":
            val_metrics[
                "balanced_accuracy"
            ],

        "val_f1":
            val_metrics["f1"],

        "val_auc":
            val_metrics["roc_auc"]
    })

    # Early stopping sulla validation loss
    if (
        val_metrics["loss"]
        <
        best_val_loss - MIN_DELTA
    ):

        best_val_loss = (
            val_metrics["loss"]
        )

        best_epoch = epoch

        best_state = copy.deepcopy(
            pilot_model.state_dict()
        )

        epochs_without_improvement = 0

    else:

        epochs_without_improvement += 1

    # Output ogni 10 epoche
    if (
        epoch == 1
        or epoch % 10 == 0
    ):

        print(
            f"Epoch {epoch:03d} | "
            f"train loss={train_loss:.4f} | "
            f"val loss={val_metrics['loss']:.4f} | "
            f"BA={val_metrics['balanced_accuracy']:.4f} | "
            f"AUC={val_metrics['roc_auc']:.4f}"
        )

    # Stop
    if (
        epochs_without_improvement
        >= PATIENCE
    ):
        print(
            f"\nEarly stopping "
            f"all'epoca {epoch}."
        )
        break

# Ripristino modello migliore
pilot_model.load_state_dict(
    best_state
)
history_df = pd.DataFrame(
    history
)
print("\n" + "=" * 70)
print(
    "MIGLIORE EPOCA:",
    best_epoch
)
print(
    "Migliore validation loss:",
    round(best_val_loss, 4)
)

best_val_metrics = evaluate_mil(
    pilot_model,
    val_loader,
    criterion,
    DEVICE
)

print("\nMetriche validation al best epoch:")
print(
    "Accuracy:",
    round(
        best_val_metrics["accuracy"],
        4
    )
)

print(
    "Balanced Accuracy:",
    round(
        best_val_metrics[
            "balanced_accuracy"
        ],
        4
    )
)

print(
    "F1:",
    round(
        best_val_metrics["f1"],
        4
    )
)

print(
    "ROC-AUC:",
    round(
        best_val_metrics["roc_auc"],
        4
    )
)

Epoch 001 | train loss=0.7115 | val loss=0.6864 | BA=0.5268 | AUC=0.5714
Epoch 010 | train loss=0.2495 | val loss=0.7278 | BA=0.6696 | AUC=0.6250
Epoch 020 | train loss=0.0781 | val loss=1.0016 | BA=0.5268 | AUC=0.4643

Early stopping all'epoca 28.

MIGLIORE EPOCA: 3
Migliore validation loss: 0.6404

Metriche validation al best epoch:
Accuracy: 0.7333
Balanced Accuracy: 0.7411
F1: 0.7143
ROC-AUC: 0.75


14.18 — Retraining outer fold 0 sui 72 pazienti completi

In [21]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

OUTER_FOLD = 0
FINAL_EPOCHS_FOLD0 = best_epoch

print(
    "Epoche selezionate dalla inner validation:",
    FINAL_EPOCHS_FOLD0
)

# Outer train / outer test
outer_train_df = (
    mil_df[
        mil_df["fold"] != OUTER_FOLD
    ]
    .copy()
)
outer_test_df = (
    mil_df[
        mil_df["fold"] == OUTER_FOLD
    ]
    .copy()
)
print(
    "\nOuter train pazienti:",
    outer_train_df["patient_id"].nunique()
)
print(
    "Outer test pazienti:",
    outer_test_df["patient_id"].nunique()
)

# 1. Preprocessing FIT sui soli 72 outer-train
imputer_final0, scaler_final0 = (
    fit_mil_preprocessor(
        outer_train_df,
        FEATURE_COLS_MIL
    )
)
outer_train_scaled = (
    transform_mil_dataframe(
        outer_train_df,
        FEATURE_COLS_MIL,
        imputer_final0,
        scaler_final0
    )
)
outer_test_scaled = (
    transform_mil_dataframe(
        outer_test_df,
        FEATURE_COLS_MIL,
        imputer_final0,
        scaler_final0
    )
)

# 2. Bag
outer_train_bags = build_patient_bags(
    outer_train_scaled,
    FEATURE_COLS_MIL
)
outer_test_bags = build_patient_bags(
    outer_test_scaled,
    FEATURE_COLS_MIL
)
print(
    "\nBag outer train:",
    len(outer_train_bags)
)
print(
    "Bag outer test:",
    len(outer_test_bags)
)

assert len(outer_train_bags) == 72
assert len(outer_test_bags) == 18

# 3. DataLoader
set_seed(42)
outer_train_loader = DataLoader(
    PatientBagDataset(
        outer_train_bags
    ),
    batch_size=8,
    shuffle=True,
    collate_fn=mil_collate_fn,
    num_workers=0
)
outer_test_loader = DataLoader(
    PatientBagDataset(
        outer_test_bags
    ),
    batch_size=8,
    shuffle=False,
    collate_fn=mil_collate_fn,
    num_workers=0
)

# 4. Pesi di classe patient-level
outer_patient_labels = (
    outer_train_df[
        [
            "patient_id",
            "dn4_label"
        ]
    ]
    .drop_duplicates("patient_id")
    ["dn4_label"]
    .to_numpy()
)
outer_class_weights_np = (
    compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=outer_patient_labels
    )
)
outer_class_weights = torch.tensor(
    outer_class_weights_np,
    dtype=torch.float32,
    device=DEVICE
)
print(
    "\nPesi classi outer train:",
    outer_class_weights
    .cpu()
    .numpy()
)

Epoche selezionate dalla inner validation: 3

Outer train pazienti: 72
Outer test pazienti: 18

Bag outer train: 72
Bag outer test: 18

Pesi classi outer train: [1.0285715 0.972973 ]


 14.19 — Modello finale fold 0

In [22]:
set_seed(42)

mil_fold0_model = GatedAttentionMIL(
    input_dim=len(FEATURE_COLS_MIL),
    instance_dim=64,
    attention_dim=32,
    classifier_dim=32,
    dropout=0.2
).to(DEVICE)

criterion_fold0 = nn.CrossEntropyLoss(
    weight=outer_class_weights
)

optimizer_fold0 = torch.optim.AdamW(
    mil_fold0_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

print(
    "Training per",
    FINAL_EPOCHS_FOLD0,
    "epoche..."
)

for epoch in range(
    1,
    FINAL_EPOCHS_FOLD0 + 1
):
    train_loss = train_one_epoch(
        mil_fold0_model,
        outer_train_loader,
        optimizer_fold0,
        criterion_fold0,
        DEVICE
    )

    print(
        f"Epoch {epoch:03d} | "
        f"train loss={train_loss:.4f}"
    )

print(
    "\nTraining fold 0 completato."
)

Training per 3 epoche...
Epoch 001 | train loss=0.6982
Epoch 002 | train loss=0.6224
Epoch 003 | train loss=0.5821

Training fold 0 completato.


14.20 — Outer test fold 0

In [23]:
fold0_test_metrics = evaluate_mil(
    mil_fold0_model,
    outer_test_loader,
    criterion_fold0,
    DEVICE
)

print(
    "\n" + "=" * 70
)
print(
    "MIL DN4 — OUTER TEST FOLD 0"
)
print(
    "=" * 70
)

print(
    "Accuracy:",
    round(
        fold0_test_metrics[
            "accuracy"
        ],
        4
    )
)
print(
    "Balanced Accuracy:",
    round(
        fold0_test_metrics[
            "balanced_accuracy"
        ],
        4
    )
)
print(
    "F1:",
    round(
        fold0_test_metrics["f1"],
        4
    )
)
print(
    "ROC-AUC:",
    round(
        fold0_test_metrics[
            "roc_auc"
        ],
        4
    )
)

# Confusion matrix
cm_fold0 = confusion_matrix(
    fold0_test_metrics["y_true"],
    fold0_test_metrics["y_pred"],
    labels=[0, 1]
)
print(
    "\nConfusion matrix:"
)
print(
    cm_fold0
)

# Classification report
print(
    "\nClassification report:"
)
print(
    classification_report(
        fold0_test_metrics["y_true"],
        fold0_test_metrics["y_pred"],
        target_names=[
            "negativo",
            "positivo"
        ],
        digits=4,
        zero_division=0
    )
)

# Predizioni dei 18 pazienti
fold0_predictions = pd.DataFrame({
    "patient_id":
        fold0_test_metrics[
            "patient_id"
        ],
    "y_true":
        fold0_test_metrics[
            "y_true"
        ],
    "y_pred":
        fold0_test_metrics[
            "y_pred"
        ],
    "prob_positive":
        fold0_test_metrics[
            "prob_positive"
        ]
})

print(
    "\nPredizioni patient-level:"
)

display(
    fold0_predictions
    .sort_values("patient_id")
    .reset_index(drop=True)
    .round(4)
)


MIL DN4 — OUTER TEST FOLD 0
Accuracy: 0.3889
Balanced Accuracy: 0.3889
F1: 0.3529
ROC-AUC: 0.358

Confusion matrix:
[[4 5]
 [6 3]]

Classification report:
              precision    recall  f1-score   support

    negativo     0.4000    0.4444    0.4211         9
    positivo     0.3750    0.3333    0.3529         9

    accuracy                         0.3889        18
   macro avg     0.3875    0.3889    0.3870        18
weighted avg     0.3875    0.3889    0.3870        18


Predizioni patient-level:


,patient_id,y_true,y_pred,prob_positive
0,4,1,1,0.5299
1,14,1,1,0.5311
2,25,0,1,0.5783
3,26,1,1,0.7489
4,36,0,1,0.5714
5,38,0,1,0.7004
6,44,0,1,0.6390
7,67,1,0,0.3112
8,72,0,1,0.5331
9,82,0,0,0.3568


14.21 — Funzione completa per un outer fold MIL

In [24]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

def run_mil_outer_fold(
    outer_fold,
    seed_base=42,
    val_size=0.20,
    batch_size=8,
    max_epochs=200,
    patience=25,
    min_delta=1e-4,
    learning_rate=1e-3,
    weight_decay=1e-4
):

    print("\n" + "=" * 80)
    print(f"MIL DN4 — OUTER FOLD {outer_fold}")
    print("=" * 80)

    fold_seed = seed_base + outer_fold

    set_seed(fold_seed)

    # 1. Outer train / test PATIENT-LEVEL
    outer_train_patients = (
        mil_patients[
            mil_patients["fold"] != outer_fold
        ]
        .copy()
        .reset_index(drop=True)
    )
    outer_test_patients = (
        mil_patients[
            mil_patients["fold"] == outer_fold
        ]
        .copy()
        .reset_index(drop=True)
    )

    # 2. Inner train / validation
    inner_train_patients, val_patients = (
        train_test_split(
            outer_train_patients,
            test_size=val_size,
            stratify=outer_train_patients["dn4_label"],
            random_state=fold_seed
        )
    )
    train_ids = set(
        inner_train_patients["patient_id"]
    )
    val_ids = set(
        val_patients["patient_id"]
    )
    test_ids = set(
        outer_test_patients["patient_id"]
    )

    assert len(train_ids & val_ids) == 0
    assert len(train_ids & test_ids) == 0
    assert len(val_ids & test_ids) == 0

    print(
        "Inner train:",
        len(inner_train_patients),
        "| validation:",
        len(val_patients),
        "| outer test:",
        len(outer_test_patients)
    )

    # 3. Dataframe turn-level
    inner_train_df = mil_df[
        mil_df["patient_id"].isin(train_ids)
    ].copy()

    val_df = mil_df[
        mil_df["patient_id"].isin(val_ids)
    ].copy()

    outer_test_df = mil_df[
        mil_df["patient_id"].isin(test_ids)
    ].copy()

    # 4. Preprocessing FIT SOLO inner train
    imputer_inner, scaler_inner = (
        fit_mil_preprocessor(
            inner_train_df,
            FEATURE_COLS_MIL
        )
    )

    inner_train_scaled = (
        transform_mil_dataframe(
            inner_train_df,
            FEATURE_COLS_MIL,
            imputer_inner,
            scaler_inner
        )
    )

    val_scaled = (
        transform_mil_dataframe(
            val_df,
            FEATURE_COLS_MIL,
            imputer_inner,
            scaler_inner
        )
    )

    # 5. Bag inner train / validation
    inner_train_bags = build_patient_bags(
        inner_train_scaled,
        FEATURE_COLS_MIL
    )
    val_bags = build_patient_bags(
        val_scaled,
        FEATURE_COLS_MIL
    )
    train_loader = DataLoader(
        PatientBagDataset(inner_train_bags),
        batch_size=batch_size,
        shuffle=True,
        collate_fn=mil_collate_fn,
        num_workers=0
    )
    val_loader = DataLoader(
        PatientBagDataset(val_bags),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=mil_collate_fn,
        num_workers=0
    )

    # 6. Class weights INNER TRAIN patient-level
    inner_labels = (
        inner_train_patients[
            "dn4_label"
        ]
        .to_numpy()
    )
    inner_weights_np = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=inner_labels
    )
    inner_weights = torch.tensor(
        inner_weights_np,
        dtype=torch.float32,
        device=DEVICE
    )

    # 7. Modello per selezione epoca
    set_seed(fold_seed)

    selection_model = GatedAttentionMIL(
        input_dim=len(FEATURE_COLS_MIL),
        instance_dim=64,
        attention_dim=32,
        classifier_dim=32,
        dropout=0.2
    ).to(DEVICE)

    criterion_inner = nn.CrossEntropyLoss(
        weight=inner_weights
    )

    optimizer_inner = torch.optim.AdamW(
        selection_model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )


    # 8. Early stopping
    best_val_loss = np.inf
    best_epoch = 1
    best_state = None

    epochs_without_improvement = 0

    for epoch in range(
        1,
        max_epochs + 1
    ):
        train_loss = train_one_epoch(
            selection_model,
            train_loader,
            optimizer_inner,
            criterion_inner,
            DEVICE
        )

        val_metrics = evaluate_mil(
            selection_model,
            val_loader,
            criterion_inner,
            DEVICE
        )

        if (
            val_metrics["loss"]
            < best_val_loss - min_delta
        ):
            best_val_loss = (
                val_metrics["loss"]
            )
            best_epoch = epoch

            best_state = copy.deepcopy(
                selection_model.state_dict()
            )
            epochs_without_improvement = 0

        else:

            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= patience
        ):
            break

    assert best_state is not None

    print(
        "Best epoch:",
        best_epoch,
        "| best val loss:",
        round(best_val_loss, 4)
    )

    # 9. RETRAIN sui 72 outer-train
    # Preprocessing viene rifatto usando tutti i 72 pazienti outer-train.
    outer_train_ids = set(
        outer_train_patients["patient_id"]
    )
    outer_train_df = mil_df[
        mil_df["patient_id"].isin(
            outer_train_ids
        )
    ].copy()

    imputer_outer, scaler_outer = (
        fit_mil_preprocessor(
            outer_train_df,
            FEATURE_COLS_MIL
        )
    )

    outer_train_scaled = (
        transform_mil_dataframe(
            outer_train_df,
            FEATURE_COLS_MIL,
            imputer_outer,
            scaler_outer
        )
    )

    outer_test_scaled = (
        transform_mil_dataframe(
            outer_test_df,
            FEATURE_COLS_MIL,
            imputer_outer,
            scaler_outer
        )
    )

    outer_train_bags = build_patient_bags(
        outer_train_scaled,
        FEATURE_COLS_MIL
    )

    outer_test_bags = build_patient_bags(
        outer_test_scaled,
        FEATURE_COLS_MIL
    )

    outer_train_loader = DataLoader(
        PatientBagDataset(outer_train_bags),
        batch_size=batch_size,
        shuffle=True,
        collate_fn=mil_collate_fn,
        num_workers=0
    )

    outer_test_loader = DataLoader(
        PatientBagDataset(outer_test_bags),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=mil_collate_fn,
        num_workers=0
    )

    # 10. Class weights sui 72 outer train
    outer_labels = (
        outer_train_patients[
            "dn4_label"
        ]
        .to_numpy()
    )

    outer_weights_np = compute_class_weight(
        class_weight="balanced",
        classes=np.array([0, 1]),
        y=outer_labels
    )

    outer_weights = torch.tensor(
        outer_weights_np,
        dtype=torch.float32,
        device=DEVICE
    )

    criterion_outer = nn.CrossEntropyLoss(
        weight=outer_weights
    )

    # 11. Modello NUOVO da zero
    set_seed(fold_seed)

    final_model = GatedAttentionMIL(
        input_dim=len(FEATURE_COLS_MIL),
        instance_dim=64,
        attention_dim=32,
        classifier_dim=32,
        dropout=0.2
    ).to(DEVICE)

    optimizer_outer = torch.optim.AdamW(
        final_model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    # 12. Training per best_epoch
    for epoch in range(
        1,
        best_epoch + 1
    ):

        final_train_loss = train_one_epoch(
            final_model,
            outer_train_loader,
            optimizer_outer,
            criterion_outer,
            DEVICE
        )

    # 13. Outer test
    test_metrics = evaluate_mil(
        final_model,
        outer_test_loader,
        criterion_outer,
        DEVICE
    )

    print(
        "Outer BA:",
        round(
            test_metrics[
                "balanced_accuracy"
            ],
            4
        ),
        "| AUC:",
        round(
            test_metrics[
                "roc_auc"
            ],
            4
        )
    )

    # 14. Predizioni patient-level
    predictions = pd.DataFrame({

        "patient_id":
            test_metrics["patient_id"],

        "fold":
            outer_fold,

        "y_true":
            test_metrics["y_true"],

        "y_pred":
            test_metrics["y_pred"],

        "prob_positive":
            test_metrics[
                "prob_positive"
            ]
    })

    fold_result = {
        "fold":
            outer_fold,
        "best_epoch":
            best_epoch,
        "best_val_loss":
            best_val_loss,
        "accuracy":
            test_metrics["accuracy"],
        "balanced_accuracy":
            test_metrics[
                "balanced_accuracy"
            ],
        "f1":
            test_metrics["f1"],
        "roc_auc":
            test_metrics["roc_auc"]
    }

    return (
        fold_result,
        predictions
    )

print(
    "Funzione outer-fold MIL definita."
)

Funzione outer-fold MIL definita.


14.22 — 5-fold outer CV MIL DN4

In [ ]:
mil_fold_results = []
mil_oof_predictions = []


for fold in range(5):
    fold_result, fold_predictions = (
        run_mil_outer_fold(
            outer_fold=fold,
            seed_base=42,
            val_size=0.20,
            batch_size=8,
            max_epochs=200,
            patience=25,
            min_delta=1e-4,
            learning_rate=1e-3,
            weight_decay=1e-4
        )
    )
    mil_fold_results.append(
        fold_result
    )
    mil_oof_predictions.append(
        fold_predictions
    )


mil_fold_results_df = pd.DataFrame(
    mil_fold_results
)
mil_oof = pd.concat(
    mil_oof_predictions,
    ignore_index=True
)

print("\n" + "=" * 80)
print("MIL DN4 — FOLD-BY-FOLD")
print("=" * 80)

display(
    mil_fold_results_df.round(4)
)

print(
    "\nPazienti OOF:",
    mil_oof["patient_id"].nunique()
)
print(
    "Duplicati:",
    mil_oof["patient_id"]
    .duplicated()
    .sum()
)

assert (
    mil_oof["patient_id"].nunique()
    == 90
)

assert (
    mil_oof["patient_id"]
    .duplicated()
    .sum()
    == 0
)


MIL DN4 — OUTER FOLD 0
Inner train: 57 | validation: 15 | outer test: 18
Best epoch: 3 | best val loss: 0.6404
Outer BA: 0.3889 | AUC: 0.358

MIL DN4 — OUTER FOLD 1
Inner train: 57 | validation: 15 | outer test: 18
Best epoch: 6 | best val loss: 0.6932
Outer BA: 0.4444 | AUC: 0.4691

MIL DN4 — OUTER FOLD 2
Inner train: 57 | validation: 15 | outer test: 18
Best epoch: 11 | best val loss: 0.6363
Outer BA: 0.5556 | AUC: 0.5556

MIL DN4 — OUTER FOLD 3
Inner train: 57 | validation: 15 | outer test: 18
Best epoch: 2 | best val loss: 0.6968
Outer BA: 0.3889 | AUC: 0.5926

MIL DN4 — OUTER FOLD 4
Inner train: 57 | validation: 15 | outer test: 18
Best epoch: 38 | best val loss: 0.493
Outer BA: 0.6 | AUC: 0.5125

MIL DN4 — FOLD-BY-FOLD


,fold,best_epoch,best_val_loss,accuracy,balanced_accuracy,f1,roc_auc
0,0,3,0.6404,0.3889,0.3889,0.3529,0.3580
1,1,6,0.6932,0.4444,0.4444,0.4444,0.4691
2,2,11,0.6363,0.5556,0.5556,0.5556,0.5556
3,3,2,0.6968,0.3889,0.3889,0.3529,0.5926
4,4,38,0.4930,0.6111,0.6000,0.6667,0.5125



Pazienti OOF: 90
Duplicati: 0


14.23 — Metriche patient-level OOF MIL

In [26]:
mil_accuracy = accuracy_score(
    mil_oof["y_true"],
    mil_oof["y_pred"]
)

mil_balanced_accuracy = (
    balanced_accuracy_score(
        mil_oof["y_true"],
        mil_oof["y_pred"]
    )
)

mil_f1 = f1_score(
    mil_oof["y_true"],
    mil_oof["y_pred"],
    pos_label=1,
    zero_division=0
)

mil_auc = roc_auc_score(
    mil_oof["y_true"],
    mil_oof["prob_positive"]
)

print("\n" + "=" * 80)
print("MIL DN4 — RISULTATI PATIENT-LEVEL OOF")
print("=" * 80)

print(
    "Accuracy:",
    round(mil_accuracy, 4)
)

print(
    "Balanced Accuracy:",
    round(
        mil_balanced_accuracy,
        4
    )
)

print(
    "F1:",
    round(mil_f1, 4)
)

print(
    "ROC-AUC:",
    round(mil_auc, 4)
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        mil_oof["y_true"],
        mil_oof["y_pred"],
        labels=[0, 1]
    )
)

print("\nClassification report:")
print(
    classification_report(
        mil_oof["y_true"],
        mil_oof["y_pred"],
        target_names=[
            "negativo",
            "positivo"
        ],
        digits=4,
        zero_division=0
    )
)

print("\nMedia fold:")

display(
    mil_fold_results_df[
        [
            "accuracy",
            "balanced_accuracy",
            "f1",
            "roc_auc"
        ]
    ]
    .agg(
        ["mean", "std"]
    )
    .round(4)
)

print("\nEpoche selezionate per fold:")

display(
    mil_fold_results_df[
        [
            "fold",
            "best_epoch",
            "best_val_loss"
        ]
    ]
    .round(4)
)


MIL DN4 — RISULTATI PATIENT-LEVEL OOF
Accuracy: 0.4778
Balanced Accuracy: 0.4778
F1: 0.4835
ROC-AUC: 0.5252

Confusion matrix:
[[21 23]
 [24 22]]

Classification report:
              precision    recall  f1-score   support

    negativo     0.4667    0.4773    0.4719        44
    positivo     0.4889    0.4783    0.4835        46

    accuracy                         0.4778        90
   macro avg     0.4778    0.4778    0.4777        90
weighted avg     0.4780    0.4778    0.4778        90


Media fold:


,accuracy,balanced_accuracy,f1,roc_auc
mean,0.4778,0.4756,0.4745,0.4976
std,0.1009,0.0973,0.1360,0.0907



Epoche selezionate per fold:


,fold,best_epoch,best_val_loss
0,0,3,0.6404
1,1,6,0.6932
2,2,11,0.6363
3,3,2,0.6968
4,4,38,0.4930


14.24 — Salvataggio run MIL seed 42

In [27]:
MIL_DIR = (
    RESULTS_DIR
    / "multiple_instance_learning_DN4"
)

MIL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


mil_fold_results_df.to_csv(
    MIL_DIR / "MIL_DN4_fold_results_seed42.csv",
    index=False
)

mil_oof.to_csv(
    MIL_DIR / "MIL_DN4_patient_level_OOF_seed42.csv",
    index=False
)


mil_summary_seed42 = pd.DataFrame([
    {
        "seed": 42,
        "n_patients": 90,
        "accuracy": mil_accuracy,
        "balanced_accuracy": mil_balanced_accuracy,
        "f1": mil_f1,
        "roc_auc": mil_auc
    }
])

mil_summary_seed42.to_csv(
    MIL_DIR / "MIL_DN4_summary_seed42.csv",
    index=False
)

print("Risultati MIL seed 42 salvati in:")
print(MIL_DIR)

display(
    mil_summary_seed42.round(4)
)

Risultati MIL seed 42 salvati in:
C:\Users\acer\Desktop\ProgettoTesi\risultati\multiple_instance_learning_DN4


,seed,n_patients,accuracy,balanced_accuracy,f1,roc_auc
0,42,90,0.4778,0.4778,0.4835,0.5252


14.25 — Stabilità MIL rispetto al seed

In [28]:
MIL_SEEDS = [
    42,
    52,
    62,
    72,
    82
]

mil_seed_results = []
mil_seed_fold_results = []


for seed in MIL_SEEDS:
    print("\n" + "#" * 90)
    print("MIL — SEED", seed)
    print("#" * 90)
    seed_oof_list = []
    seed_fold_list = []

    for fold in range(5):
        fold_result, fold_predictions = (
            run_mil_outer_fold(
                outer_fold=fold,
                seed_base=seed,
                val_size=0.20,
                batch_size=8,
                max_epochs=200,
                patience=25,
                min_delta=1e-4,
                learning_rate=1e-3,
                weight_decay=1e-4
            )
        )
        seed_fold_list.append(
            fold_result
        )
        seed_oof_list.append(
            fold_predictions
        )

    # OOF sui 90 pazienti per questo seed
    seed_oof = pd.concat(
        seed_oof_list,
        ignore_index=True
    )
    assert (
        seed_oof["patient_id"].nunique()
        == 90
    )
    assert (
        seed_oof["patient_id"]
        .duplicated()
        .sum()
        == 0
    )
    seed_accuracy = accuracy_score(
        seed_oof["y_true"],
        seed_oof["y_pred"]
    )
    seed_ba = balanced_accuracy_score(
        seed_oof["y_true"],
        seed_oof["y_pred"]
    )
    seed_f1 = f1_score(
        seed_oof["y_true"],
        seed_oof["y_pred"],
        pos_label=1,
        zero_division=0
    )
    seed_auc = roc_auc_score(
        seed_oof["y_true"],
        seed_oof["prob_positive"]
    )
    mil_seed_results.append({
        "seed": seed,
        "accuracy": seed_accuracy,
        "balanced_accuracy": seed_ba,
        "f1": seed_f1,
        "roc_auc": seed_auc
    })

    # Fold results
    seed_fold_df = pd.DataFrame(
        seed_fold_list
    )
    seed_fold_df["seed"] = seed
    mil_seed_fold_results.append(
        seed_fold_df
    )
    
    print("\nRISULTATO SEED", seed)
    print(
        "Accuracy:",
        round(seed_accuracy, 4)
    )
    print(
        "Balanced Accuracy:",
        round(seed_ba, 4)
    )
    print(
        "F1:",
        round(seed_f1, 4)
    )
    print(
        "ROC-AUC:",
        round(seed_auc, 4)
    )

# Riepilogo
mil_seed_results_df = pd.DataFrame(
    mil_seed_results
)

mil_seed_fold_results_df = pd.concat(
    mil_seed_fold_results,
    ignore_index=True
)

print("\n" + "=" * 80)
print("MIL DN4 — STABILITÀ SU 5 SEED")
print("=" * 80)

display(
    mil_seed_results_df.round(4)
)

print("\nMedia ± std tra seed:")

display(
    mil_seed_results_df[
        [
            "accuracy",
            "balanced_accuracy",
            "f1",
            "roc_auc"
        ]
    ]
    .agg(
        ["mean", "std", "min", "max"]
    )
    .round(4)
)


##########################################################################################
MIL — SEED 42
##########################################################################################

MIL DN4 — OUTER FOLD 0
Inner train: 57 | validation: 15 | outer test: 18
Best epoch: 3 | best val loss: 0.6404
Outer BA: 0.3889 | AUC: 0.358

MIL DN4 — OUTER FOLD 1
Inner train: 57 | validation: 15 | outer test: 18
Best epoch: 6 | best val loss: 0.6932
Outer BA: 0.4444 | AUC: 0.4691

MIL DN4 — OUTER FOLD 2
Inner train: 57 | validation: 15 | outer test: 18
Best epoch: 11 | best val loss: 0.6363
Outer BA: 0.5556 | AUC: 0.5556

MIL DN4 — OUTER FOLD 3
Inner train: 57 | validation: 15 | outer test: 18
Best epoch: 2 | best val loss: 0.6968
Outer BA: 0.3889 | AUC: 0.5926

MIL DN4 — OUTER FOLD 4
Inner train: 57 | validation: 15 | outer test: 18
Best epoch: 38 | best val loss: 0.493
Outer BA: 0.6 | AUC: 0.5125

RISULTATO SEED 42
Accuracy: 0.4778
Balanced Accuracy: 0.4778
F1: 0.4835
ROC-AUC: 0.5252

#

,seed,accuracy,balanced_accuracy,f1,roc_auc
0,42,0.4778,0.4778,0.4835,0.5252
1,52,0.5222,0.5237,0.4941,0.5208
2,62,0.4889,0.4896,0.4773,0.5104
3,72,0.5111,0.5128,0.4762,0.5509
4,82,0.4889,0.4872,0.5306,0.5469



Media ± std tra seed:


,accuracy,balanced_accuracy,f1,roc_auc
mean,0.4978,0.4982,0.4923,0.5308
std,0.0183,0.0192,0.0225,0.0174
min,0.4778,0.4778,0.4762,0.5104
max,0.5222,0.5237,0.5306,0.5509


14.26 — Salvataggio finale MIL multi-seed

In [ ]:
MIL_DIR = (
    RESULTS_DIR
    / "multiple_instance_learning_DN4"
)

MIL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Risultati per seed
mil_seed_results_df.to_csv(
    MIL_DIR / "MIL_DN4_stabilita_5_seed.csv",
    index=False
)

# Risultati fold-by-fold di tutti i seed
mil_seed_fold_results_df.to_csv(
    MIL_DIR / "MIL_DN4_fold_results_5_seed.csv",
    index=False
)

# Riepilogo statistico
mil_seed_stats = (
    mil_seed_results_df[
        [
            "accuracy",
            "balanced_accuracy",
            "f1",
            "roc_auc"
        ]
    ]
    .agg(
        ["mean", "std", "min", "max"]
    )
)

mil_seed_stats.to_csv(
    MIL_DIR / "MIL_DN4_stabilita_statistiche.csv"
)

# Confronto finale dei tre approcci DN4
dn4_final_comparison = pd.DataFrame([
    {
        "method": "Random Forest turn-level",
        "balanced_accuracy": 0.5761,
        "roc_auc": 0.5731,
        "note": "5-fold patient-level CV"
    },

    {
        "method": "XGBoost turn-level",
        "balanced_accuracy": 0.5988,
        "roc_auc": 0.6117,
        "note": (
            "5-fold patient-level CV; "
            "permutation p_BA=0.0769, p_AUC=0.0829"
        )
    },

    {
        "method": "MIL gated attention",
        "balanced_accuracy":
            mil_seed_results_df[
                "balanced_accuracy"
            ].mean(),

        "roc_auc":
            mil_seed_results_df[
                "roc_auc"
            ].mean(),

        "note":
            "mean across 5 seeds"
    }
])

dn4_final_comparison.to_csv(
    MIL_DIR / "confronto_finale_DN4_RF_XGBoost_MIL.csv",
    index=False
)

print("Risultati salvati in:")
print(MIL_DIR)
print("\nConfronto finale DN4:")
display(
    dn4_final_comparison.round(4)
)

print("\nStabilità MIL:")
display(
    mil_seed_stats.round(4)
)

Risultati salvati in:
C:\Users\acer\Desktop\ProgettoTesi\risultati\multiple_instance_learning_DN4

Confronto finale DN4:


,method,balanced_accuracy,roc_auc,note
0,Random Forest turn-level,0.5761,0.5731,5-fold patient-level CV
1,XGBoost turn-level,0.5988,0.6117,5-fold patient-level CV; permutation p_BA=0.07...
2,MIL gated attention,0.4982,0.5308,mean across 5 seeds



Stabilità MIL:


,accuracy,balanced_accuracy,f1,roc_auc
mean,0.4978,0.4982,0.4923,0.5308
std,0.0183,0.0192,0.0225,0.0174
min,0.4778,0.4778,0.4762,0.5104
max,0.5222,0.5237,0.5306,0.5509
